# Colab ortam kontrolü — v3

**Python 3 / CPU / Runtime Version 2026.07** seçin.
Bu sürümde cgroup bilgisi okunamazsa kaynak durumu `unavailable` kalır; diğer kontroller tamamlanır.
Gömülü proje kaynakları SHA-256 ile doğrulanır. Paket kurulumu veya medya/model indirmesi yoktur.

GPU sorgusu varsayılan olarak kapalıdır. Bu notebook GPU oturumu açmaz, model oluşturmaz ve eğitim/ölçüm çalıştırmaz.
Tamamlanan önceki GPU notebook'larını yeniden çalıştırmayın.


In [ ]:
import base64
import hashlib
import json
import os
import sys
from pathlib import Path

SOURCE_SHA256 = "3836567a8108e41df80ce5f30bcc392ceee568337e3e462d15aa14741875e47c"
BASE_COMMIT = "2f41bfe2e95febcaab80f80ab4875c72073c3a32"
SOURCE_B64 = (
    'ewogICJiYXNlX2NvbW1pdCI6ICIyZjQxYmZlMmU5NWZlYmNhYWI4MGY4MGFiNDg3NWM3MjA3M2MzYTMyIiwKICAi'
    'ZmlsZXMiOiB7CiAgICAiTElDRU5TRSI6IHsKICAgICAgInNoYTI1NiI6ICI3OGJkNmUwZDM2ODI1MjU2YzYzYzY5'
    'ZmVlMWQxMGVmOGE4ZTczM2U0NzgwNzlkNzNmODI0NjdkOWM0ZTI4MTQyIiwKICAgICAgInV0ZjgiOiAiTUlUIExp'
    'Y2Vuc2VcblxuQ29weXJpZ2h0IChjKSAyMDI2IGVya2FucnpnY1xuXG5QZXJtaXNzaW9uIGlzIGhlcmVieSBncmFu'
    'dGVkLCBmcmVlIG9mIGNoYXJnZSwgdG8gYW55IHBlcnNvbiBvYnRhaW5pbmcgYSBjb3B5XG5vZiB0aGlzIHNvZnR3'
    'YXJlIGFuZCBhc3NvY2lhdGVkIGRvY3VtZW50YXRpb24gZmlsZXMgKHRoZSBcIlNvZnR3YXJlXCIpLCB0byBkZWFs'
    'XG5pbiB0aGUgU29mdHdhcmUgd2l0aG91dCByZXN0cmljdGlvbiwgaW5jbHVkaW5nIHdpdGhvdXQgbGltaXRhdGlv'
    'biB0aGUgcmlnaHRzXG50byB1c2UsIGNvcHksIG1vZGlmeSwgbWVyZ2UsIHB1Ymxpc2gsIGRpc3RyaWJ1dGUsIHN1'
    'YmxpY2Vuc2UsIGFuZC9vciBzZWxsXG5jb3BpZXMgb2YgdGhlIFNvZnR3YXJlLCBhbmQgdG8gcGVybWl0IHBlcnNv'
    'bnMgdG8gd2hvbSB0aGUgU29mdHdhcmUgaXNcbmZ1cm5pc2hlZCB0byBkbyBzbywgc3ViamVjdCB0byB0aGUgZm9s'
    'bG93aW5nIGNvbmRpdGlvbnM6XG5cblRoZSBhYm92ZSBjb3B5cmlnaHQgbm90aWNlIGFuZCB0aGlzIHBlcm1pc3Np'
    'b24gbm90aWNlIHNoYWxsIGJlIGluY2x1ZGVkIGluIGFsbFxuY29waWVzIG9yIHN1YnN0YW50aWFsIHBvcnRpb25z'
    'IG9mIHRoZSBTb2Z0d2FyZS5cblxuVEhFIFNPRlRXQVJFIElTIFBST1ZJREVEIFwiQVMgSVNcIiwgV0lUSE9VVCBX'
    'QVJSQU5UWSBPRiBBTlkgS0lORCwgRVhQUkVTUyBPUlxuSU1QTElFRCwgSU5DTFVESU5HIEJVVCBOT1QgTElNSVRF'
    'RCBUTyBUSEUgV0FSUkFOVElFUyBPRiBNRVJDSEFOVEFCSUxJVFksXG5GSVRORVNTIEZPUiBBIFBBUlRJQ1VMQVIg'
    'UFVSUE9TRSBBTkQgTk9OSU5GUklOR0VNRU5ULiBJTiBOTyBFVkVOVCBTSEFMTCBUSEVcbkFVVEhPUlMgT1IgQ09Q'
    'WVJJR0hUIEhPTERFUlMgQkUgTElBQkxFIEZPUiBBTlkgQ0xBSU0sIERBTUFHRVMgT1IgT1RIRVJcbkxJQUJJTElU'
    'WSwgV0hFVEhFUiBJTiBBTiBBQ1RJT04gT0YgQ09OVFJBQ1QsIFRPUlQgT1IgT1RIRVJXSVNFLCBBUklTSU5HIEZS'
    'T00sXG5PVVQgT0YgT1IgSU4gQ09OTkVDVElPTiBXSVRIIFRIRSBTT0ZUV0FSRSBPUiBUSEUgVVNFIE9SIE9USEVS'
    'IERFQUxJTkdTIElOIFRIRVxuU09GVFdBUkUuXG4iCiAgICB9LAogICAgImNvcmUvX19pbml0X18ucHkiOiB7CiAg'
    'ICAgICJzaGEyNTYiOiAiZTNiMGM0NDI5OGZjMWMxNDlhZmJmNGM4OTk2ZmI5MjQyN2FlNDFlNDY0OWI5MzRjYTQ5'
    'NTk5MWI3ODUyYjg1NSIsCiAgICAgICJ1dGY4IjogIiIKICAgIH0sCiAgICAiY29yZS9jb2xhYl9hcnRpZmFjdHMu'
    'cHkiOiB7CiAgICAgICJzaGEyNTYiOiAiYzgxZjlkMDk4NzMwODM0OTBhODBmOWMyMWFjOWUwMTYxODc1NGM0MDU0'
    'ZjdiZTBmMThmMWI0NzgzM2Q5MjdkZiIsCiAgICAgICJ1dGY4IjogIlwiXCJcIlNtYWxsIGJvdW5kZWQsIGV4Y2x1'
    'c2l2ZSBhcnRpZmFjdHMgZm9yIHRoZSBleHBsaWNpdCBDb2xhYiBwcmVwYXJhdGlvbiBwYXRoLlwiXCJcIlxuXG5m'
    'cm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zXG5cbmltcG9ydCBoYXNobGliXG5pbXBvcnQganNvblxu'
    'aW1wb3J0IG9zXG5pbXBvcnQgcmVcbmltcG9ydCBzdGF0XG5mcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGhcblxuTUlC'
    'ID0gMTAyNCoqMlxuXG5cbmRlZiBkaWdlc3QocmF3OiBieXRlcykgLT4gc3RyOlxuICAgIHJldHVybiBoYXNobGli'
    'LnNoYTI1NihyYXcpLmhleGRpZ2VzdCgpXG5cblxuZGVmIGlkZW50aXR5KHZhbHVlOiBzdHIsIHNpemU6IGludCA9'
    'IDY0KSAtPiBzdHI6XG4gICAgaWYgbm90IGlzaW5zdGFuY2UodmFsdWUsIHN0cikgb3Igbm90IHJlLmZ1bGxtYXRj'
    'aChmXCJbMC05YS1mXXt7e3NpemV9fX1cIiwgdmFsdWUpOlxuICAgICAgICByYWlzZSBWYWx1ZUVycm9yKFwiaW52'
    'YWxpZCBkaWdlc3RcIilcbiAgICByZXR1cm4gdmFsdWVcblxuXG5kZWYgc2FmZShwYXRoOiBQYXRoKSAtPiBOb25l'
    'OlxuICAgIGlmIGFueShwLmlzX3N5bWxpbmsoKSBmb3IgcCBpbiAocGF0aCwgKnBhdGgucGFyZW50cykpOlxuICAg'
    'ICAgICByYWlzZSBWYWx1ZUVycm9yKFwic3ltbGluayBhcnRpZmFjdFwiKVxuXG5cbmRlZiByZWFkKHBhdGg6IFBh'
    'dGgsIGxpbWl0OiBpbnQpIC0+IGJ5dGVzOlxuICAgIHNhZmUocGF0aClcbiAgICBmZCA9IG9zLm9wZW4ocGF0aCwg'
    'b3MuT19SRE9OTFkgfCBvcy5PX05PRk9MTE9XIHwgb3MuT19OT05CTE9DSylcbiAgICB3aXRoIG9zLmZkb3Blbihm'
    'ZCwgXCJyYlwiKSBhcyBzdHJlYW06XG4gICAgICAgIGlmIG5vdCBzdGF0LlNfSVNSRUcob3MuZnN0YXQoc3RyZWFt'
    'LmZpbGVubygpKS5zdF9tb2RlKTpcbiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoXCJub25yZWd1bGFyIGFy'
    'dGlmYWN0XCIpXG4gICAgICAgIHJhdyA9IHN0cmVhbS5yZWFkKGxpbWl0ICsgMSlcbiAgICBpZiBub3QgcmF3IG9y'
    'IGxlbihyYXcpID4gbGltaXQ6XG4gICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoXCJhcnRpZmFjdCBieXRlIGxpbWl0'
    'XCIpXG4gICAgcmV0dXJuIHJhd1xuXG5cbmRlZiBlbmNvZGUodmFsdWU6IGRpY3QpIC0+IGJ5dGVzOlxuICAgIHJl'
    'dHVybiAoanNvbi5kdW1wcyh2YWx1ZSwgc29ydF9rZXlzPVRydWUsIGluZGVudD0yLCBhbGxvd19uYW49RmFsc2Up'
    'ICsgXCJcXG5cIikuZW5jb2RlKClcblxuXG5kZWYgZG9jdW1lbnQocGF0aDogUGF0aCwgbGltaXQ6IGludCA9IE1J'
    'QikgLT4gZGljdDpcbiAgICB2YWx1ZSA9IGpzb24ubG9hZHMocmVhZChwYXRoLCBsaW1pdCkpXG4gICAgaWYgbm90'
    'IGlzaW5zdGFuY2UodmFsdWUsIGRpY3QpOlxuICAgICAgICByYWlzZSBWYWx1ZUVycm9yKFwiYXJ0aWZhY3Qgb2Jq'
    'ZWN0IHJlcXVpcmVkXCIpXG4gICAgcmV0dXJuIHZhbHVlXG5cblxuZGVmIHB1dChwYXRoOiBQYXRoLCByYXc6IGJ5'
    'dGVzLCBsaW1pdDogaW50ID0gTUlCKSAtPiBzdHI6XG4gICAgc2FmZShwYXRoKVxuICAgIGlmIG5vdCByYXcgb3Ig'
    'bGVuKHJhdykgPiBsaW1pdDpcbiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihcIm91dHB1dCBieXRlIGxpbWl0XCIp'
    'XG4gICAgd2l0aCBwYXRoLm9wZW4oXCJ4YlwiKSBhcyBzdHJlYW06XG4gICAgICAgIHN0cmVhbS53cml0ZShyYXcp'
    'XG4gICAgICAgIHN0cmVhbS5mbHVzaCgpXG4gICAgICAgIG9zLmZzeW5jKHN0cmVhbS5maWxlbm8oKSlcbiAgICBy'
    'ZXR1cm4gZGlnZXN0KHJhdylcblxuXG5kZWYgZGlyZWN0b3J5KHBhdGg6IFBhdGgpIC0+IE5vbmU6XG4gICAgc2Fm'
    'ZShwYXRoKVxuICAgIHBhdGgubWtkaXIoKSAgIyBFeGlzdGluZyBhdHRlbXB0cyBhcmUgY29uc3VtZWQsIGluY2x1'
    'ZGluZyBpbmNvbXBsZXRlIG9uZXMuXG4iCiAgICB9LAogICAgImNvcmUvc3BhdGlhbF9jb2xhYl9yZWFkaW5lc3Mu'
    'cHkiOiB7CiAgICAgICJzaGEyNTYiOiAiMmM1NmMwZDE1NTdhZGY5MGVkNDk4ZTE3ZTRjZGRkZTcwOWU5ZDBlNWRh'
    'NjRkNGMyZjk4Nzk5N2M2OTU2OGFlOCIsCiAgICAgICJ1dGY4IjogIlwiXCJcIlJlYWQtb25seSBDb2xhYiByZWFk'
    'aW5lc3M7IGhhcmR3YXJlIHZpc2liaWxpdHkgbmV2ZXIgZXN0YWJsaXNoZXMgcmVzb3VyY2UgY29udHJvbC5cIlwi'
    'XCJcblxuZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9uc1xuXG5pbXBvcnQgaW1wb3J0bGliLm1ldGFk'
    'YXRhXG5pbXBvcnQgb3NcbmltcG9ydCBwbGF0Zm9ybVxuaW1wb3J0IHJlXG5pbXBvcnQgcmVzb3VyY2VcbmltcG9y'
    'dCBzaHV0aWxcbmltcG9ydCBzdWJwcm9jZXNzXG5pbXBvcnQgdGVtcGZpbGVcbmltcG9ydCB0aW1lXG5mcm9tIGRh'
    'dGFjbGFzc2VzIGltcG9ydCBhc2RpY3QsIGRhdGFjbGFzc1xuZnJvbSBwYXRobGliIGltcG9ydCBQYXRoLCBQdXJl'
    'UG9zaXhQYXRoXG5cbmZyb20gY29yZSBpbXBvcnQgY29sYWJfYXJ0aWZhY3RzIGFzIGFcblxuTUFYX1NFQ09ORFMg'
    'PSA2MFxuTUFYX09VVFBVVF9CWVRFUyA9IDY1NTM2XG5QQUNLQUdFUyA9IChcIm51bXB5XCIsIFwiUGlsbG93XCIs'
    'IFwidG9yY2hcIiwgXCJvbm54XCIsIFwib25ueHJ1bnRpbWVcIilcbkVYUEVDVEVEID0ge1wicHl0aG9uXCI6IFwi'
    'My4xMi4xM1wiLCBcIm51bXB5XCI6IFwiMi4wLjJcIiwgXCJQaWxsb3dcIjogXCIxMS4zLjBcIn1cbkNPVU5URVJT'
    'ID0ge1xuICAgIFwidjJcIjogKFwibWVtb3J5Lm1heFwiLCBcIm1lbW9yeS5zd2FwLm1heFwiLCBcIm1lbW9yeS5w'
    'ZWFrXCIsIFwibWVtb3J5LmN1cnJlbnRcIiksXG4gICAgXCJ2MVwiOiAoXG4gICAgICAgIFwibWVtb3J5LmxpbWl0'
    'X2luX2J5dGVzXCIsIFwibWVtb3J5Lm1lbXN3LmxpbWl0X2luX2J5dGVzXCIsXG4gICAgICAgIFwibWVtb3J5Lm1h'
    'eF91c2FnZV9pbl9ieXRlc1wiLCBcIm1lbW9yeS51c2FnZV9pbl9ieXRlc1wiLFxuICAgICksXG59XG5cblxuQGRh'
    'dGFjbGFzcyhmcm96ZW49VHJ1ZSlcbmNsYXNzIFJlc291cmNlTWV0YWRhdGE6XG4gICAgc3RhdHVzOiBzdHJcbiAg'
    'ICByZWFzb246IHN0clxuICAgIHZlcnNpb246IHN0ciB8IE5vbmUgPSBOb25lXG4gICAgY291bnRlcnM6IHR1cGxl'
    'W3R1cGxlW3N0ciwgaW50IHwgTm9uZV0sIC4uLl0gPSAoKVxuXG5cbkBkYXRhY2xhc3MoZnJvemVuPVRydWUpXG5j'
    'bGFzcyBHcHVNZXRhZGF0YTpcbiAgICBzdGF0dXM6IHN0clxuICAgIHJlYXNvbjogc3RyXG4gICAgbmFtZTogc3Ry'
    'IHwgTm9uZSA9IE5vbmVcbiAgICB2cmFtX21pYjogaW50IHwgTm9uZSA9IE5vbmVcbiAgICBkcml2ZXI6IHN0ciB8'
    'IE5vbmUgPSBOb25lXG5cblxuZGVmIHJlc291cmNlcyhcbiAgICAqLCBwcm9jX3Jvb3Q6IFBhdGggPSBQYXRoKFwi'
    'L3Byb2NcIiksIGNncm91cF9yb290OiBQYXRoID0gUGF0aChcIi9zeXMvZnMvY2dyb3VwXCIpLFxuICAgIHBpZDog'
    'aW50IHwgTm9uZSA9IE5vbmUsXG4pIC0+IFJlc291cmNlTWV0YWRhdGE6XG4gICAgXCJcIlwiUmVmdXNlIGluYWNj'
    'ZXNzaWJsZSBuYW1lc3BhY2UgaWRlbnRpdGllczsgbmV2ZXIgbm9ybWFsaXplIHBhcmVudCBzZWdtZW50cy5cIlwi'
    'XCJcbiAgICBwcm9jZXNzID0gb3MuZ2V0cGlkKCkgaWYgcGlkIGlzIE5vbmUgZWxzZSBwaWRcbiAgICBpZiB0eXBl'
    'KHByb2Nlc3MpIGlzIG5vdCBpbnQgb3Igbm90IDAgPCBwcm9jZXNzIDwgMioqMzE6XG4gICAgICAgIHJhaXNlIFZh'
    'bHVlRXJyb3IoXCJwb3NpdGl2ZSBib3VuZGVkIHByb2Nlc3MgaWRlbnRpdHkgcmVxdWlyZWRcIilcbiAgICB0cnk6'
    'XG4gICAgICAgIHJhdyA9IGEucmVhZChwcm9jX3Jvb3QgLyBzdHIocHJvY2VzcykgLyBcImNncm91cFwiLCA0MDk2'
    'KS5kZWNvZGUoKVxuICAgICAgICBjYW5kaWRhdGVzID0gW11cbiAgICAgICAgZm9yIGxpbmUgaW4gcmF3LnNwbGl0'
    'bGluZXMoKTpcbiAgICAgICAgICAgIGZpZWxkcyA9IGxpbmUuc3BsaXQoXCI6XCIsIDIpXG4gICAgICAgICAgICBp'
    'ZiBsZW4oZmllbGRzKSAhPSAzIG9yIG5vdCBmaWVsZHNbMF0uaXNhc2NpaSgpIG9yIG5vdCBmaWVsZHNbMF0uaXNk'
    'ZWNpbWFsKCk6XG4gICAgICAgICAgICAgICAgcmV0dXJuIFJlc291cmNlTWV0YWRhdGEoXCJ1bmF2YWlsYWJsZVwi'
    'LCBcIm1hbGZvcm1lZF9pZGVudGl0eVwiKVxuICAgICAgICAgICAgaGllcmFyY2h5LCBjb250cm9sbGVycywgcmVs'
    'YXRpdmUgPSBmaWVsZHNcbiAgICAgICAgICAgIHZlcnNpb24gPSBcInYxXCIgaWYgXCJtZW1vcnlcIiBpbiBjb250'
    'cm9sbGVycy5zcGxpdChcIixcIikgZWxzZSBcInYyXCJcbiAgICAgICAgICAgIGlmIHZlcnNpb24gPT0gXCJ2Mlwi'
    'IGFuZCAoaGllcmFyY2h5ICE9IFwiMFwiIG9yIGNvbnRyb2xsZXJzKTpcbiAgICAgICAgICAgICAgICBjb250aW51'
    'ZVxuICAgICAgICAgICAgcGF0aCA9IFB1cmVQb3NpeFBhdGgocmVsYXRpdmUpXG4gICAgICAgICAgICBpZiBcIi4u'
    'XCIgaW4gcGF0aC5wYXJ0czpcbiAgICAgICAgICAgICAgICByZXR1cm4gUmVzb3VyY2VNZXRhZGF0YShcInVuYXZh'
    'aWxhYmxlXCIsIFwicGFyZW50X2lkZW50aXR5XCIsIHZlcnNpb24pXG4gICAgICAgICAgICBpZiBub3QgcGF0aC5p'
    'c19hYnNvbHV0ZSgpIG9yIGxlbihwYXRoLnBhcnRzKSA+IDMyIG9yIFwiXFx4MDBcIiBpbiByZWxhdGl2ZTpcbiAg'
    'ICAgICAgICAgICAgICByZXR1cm4gUmVzb3VyY2VNZXRhZGF0YShcInVuYXZhaWxhYmxlXCIsIFwibWFsZm9ybWVk'
    'X2lkZW50aXR5XCIsIHZlcnNpb24pXG4gICAgICAgICAgICBmb2xkZXIgPSBjZ3JvdXBfcm9vdCAvIFwibWVtb3J5'
    'XCIgaWYgdmVyc2lvbiA9PSBcInYxXCIgZWxzZSBjZ3JvdXBfcm9vdFxuICAgICAgICAgICAgY2FuZGlkYXRlcy5h'
    'cHBlbmQoKHZlcnNpb24sIGZvbGRlci5qb2lucGF0aCgqcGF0aC5wYXJ0c1sxOl0pKSlcbiAgICAgICAgaWYgbGVu'
    'KGNhbmRpZGF0ZXMpICE9IDE6XG4gICAgICAgICAgICByZXR1cm4gUmVzb3VyY2VNZXRhZGF0YShcInVuYXZhaWxh'
    'YmxlXCIsIFwiYW1iaWd1b3VzX29yX21pc3NpbmdfY29udHJvbGxlclwiKVxuICAgICAgICB2ZXJzaW9uLCBmb2xk'
    'ZXIgPSBjYW5kaWRhdGVzWzBdXG4gICAgICAgIGNvdW50ZXJzID0gW11cbiAgICAgICAgZm9yIG5hbWUgaW4gQ09V'
    'TlRFUlNbdmVyc2lvbl06XG4gICAgICAgICAgICB2YWx1ZSA9IGEucmVhZChmb2xkZXIgLyBuYW1lLCAzMikuZGVj'
    'b2RlKCkuc3RyaXAoKVxuICAgICAgICAgICAgaWYgdmFsdWUgPT0gXCJtYXhcIjpcbiAgICAgICAgICAgICAgICBu'
    'dW1iZXIgPSBOb25lXG4gICAgICAgICAgICBlbGlmIHZhbHVlLmlzYXNjaWkoKSBhbmQgdmFsdWUuaXNkZWNpbWFs'
    'KCkgYW5kIGxlbih2YWx1ZSkgPD0gMjA6XG4gICAgICAgICAgICAgICAgbnVtYmVyID0gaW50KHZhbHVlKVxuICAg'
    'ICAgICAgICAgICAgIG51bWJlciA9IG51bWJlciBpZiBudW1iZXIgPCAyKio2MCBlbHNlIE5vbmVcbiAgICAgICAg'
    'ICAgIGVsc2U6XG4gICAgICAgICAgICAgICAgcmV0dXJuIFJlc291cmNlTWV0YWRhdGEoXCJ1bmF2YWlsYWJsZVwi'
    'LCBcIm1hbGZvcm1lZF9jb3VudGVyXCIsIHZlcnNpb24pXG4gICAgICAgICAgICBjb3VudGVycy5hcHBlbmQoKG5h'
    'bWUsIG51bWJlcikpXG4gICAgICAgIHJldHVybiBSZXNvdXJjZU1ldGFkYXRhKFxuICAgICAgICAgICAgXCJvYnNl'
    'cnZlZF9tZXRhZGF0YV9vbmx5XCIsIFwibm90X2VuZm9yY2VtZW50XCIsIHZlcnNpb24sIHR1cGxlKGNvdW50ZXJz'
    'KVxuICAgICAgICApXG4gICAgZXhjZXB0IChPU0Vycm9yLCBVbmljb2RlRXJyb3IsIFZhbHVlRXJyb3IpOlxuICAg'
    'ICAgICByZXR1cm4gUmVzb3VyY2VNZXRhZGF0YShcInVuYXZhaWxhYmxlXCIsIFwiaW5hY2Nlc3NpYmxlX29yX3Vu'
    'c2FmZV9tZXRhZGF0YVwiKVxuXG5cbmRlZiBncHUoKiwgd29ya2RpcjogUGF0aCB8IE5vbmUgPSBOb25lKSAtPiBH'
    'cHVNZXRhZGF0YTpcbiAgICBcIlwiXCJPbmUgZml4ZWQsIGJvdW5kZWQgTlZJRElBIHF1ZXJ5OyBubyBUb3JjaCBp'
    'bXBvcnQgb3IgdGVuc29yIG9wZXJhdGlvbi5cIlwiXCJcbiAgICBpZiB3b3JrZGlyIGlzIG5vdCBOb25lOlxuICAg'
    'ICAgICBhLnNhZmUod29ya2RpcilcbiAgICBsaW1pdGVyLCBiaW5hcnkgPSBzaHV0aWwud2hpY2goXCJwcmxpbWl0'
    'XCIpLCBzaHV0aWwud2hpY2goXCJudmlkaWEtc21pXCIpXG4gICAgaWYgbGltaXRlciBpcyBOb25lIG9yIGJpbmFy'
    'eSBpcyBOb25lOlxuICAgICAgICByZXR1cm4gR3B1TWV0YWRhdGEoXCJ1bmF2YWlsYWJsZVwiLCBcIm1pc3Npbmdf'
    'cXVlcnlfdG9vbFwiKVxuICAgIGFyZ3VtZW50cyA9IFtcbiAgICAgICAgbGltaXRlciwgXCItLWZzaXplPTgxOTI6'
    'ODE5MlwiLCBcIi0tY3B1PTU6NVwiLCBcIi0tXCIsIGJpbmFyeSxcbiAgICAgICAgXCItLXF1ZXJ5LWdwdT1uYW1l'
    'LG1lbW9yeS50b3RhbCxkcml2ZXJfdmVyc2lvblwiLCBcIi0tZm9ybWF0PWNzdixub2hlYWRlcixub3VuaXRzXCIs'
    'XG4gICAgXVxuICAgIHRyeTpcbiAgICAgICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlGaWxlKGRpcj13b3JrZGly'
    'KSBhcyBjYXB0dXJlOlxuICAgICAgICAgICAgcHJvY2VzcyA9IHN1YnByb2Nlc3MucnVuKCAgIyBub3FhOiBTNjAz'
    'IC0tIGZpeGVkIHN5c3RlbSBxdWVyeSBhbmQga2VybmVsIGxpbWl0c1xuICAgICAgICAgICAgICAgIGFyZ3VtZW50'
    'cywgc3Rkb3V0PWNhcHR1cmUsIHN0ZGVycj1zdWJwcm9jZXNzLlNURE9VVCwgdGltZW91dD0xMCwgY2hlY2s9RmFs'
    'c2UsXG4gICAgICAgICAgICApXG4gICAgICAgICAgICBjYXB0dXJlLnNlZWsoMClcbiAgICAgICAgICAgIHJhdyA9'
    'IGNhcHR1cmUucmVhZCg4MTkzKVxuICAgICAgICBpZiBwcm9jZXNzLnJldHVybmNvZGUgb3IgbGVuKHJhdykgPiA4'
    'MTkyOlxuICAgICAgICAgICAgcmV0dXJuIEdwdU1ldGFkYXRhKFwidW5hdmFpbGFibGVcIiwgXCJxdWVyeV9mYWls'
    'ZWRfb3Jfb3ZlcnNpemVkXCIpXG4gICAgICAgIGxpbmVzID0gcmF3LmRlY29kZShcImFzY2lpXCIpLnN0cmlwKCku'
    'c3BsaXRsaW5lcygpXG4gICAgICAgIGlmIGxlbihsaW5lcykgIT0gMTpcbiAgICAgICAgICAgIHJldHVybiBHcHVN'
    'ZXRhZGF0YShcInVuYXZhaWxhYmxlXCIsIFwic2luZ2xlX2dwdV9yZXF1aXJlZFwiKVxuICAgICAgICBmaWVsZHMg'
    'PSBbdGV4dC5zdHJpcCgpIGZvciB0ZXh0IGluIGxpbmVzWzBdLnNwbGl0KFwiLFwiKV1cbiAgICAgICAgaWYgKFxu'
    'ICAgICAgICAgICAgbGVuKGZpZWxkcykgIT0gMyBvciBub3QgMCA8IGxlbihmaWVsZHNbMF0pIDw9IDEyOFxuICAg'
    'ICAgICAgICAgb3Igbm90IGFsbCgzMiA8PSBvcmQoYykgPCAxMjcgZm9yIGMgaW4gZmllbGRzWzBdKVxuICAgICAg'
    'ICAgICAgb3Igbm90IGZpZWxkc1sxXS5pc2RpZ2l0KCkgb3IgbGVuKGZpZWxkc1sxXSkgPiA2XG4gICAgICAgICAg'
    'ICBvciBub3QgMCA8IGludChmaWVsZHNbMV0pIDw9IDUyNDI4OFxuICAgICAgICAgICAgb3IgcmUuZnVsbG1hdGNo'
    'KHJcIlswLTldezEsNH0oPzpcXC5bMC05XXsxLDR9KXsxLDN9XCIsIGZpZWxkc1syXSkgaXMgTm9uZVxuICAgICAg'
    'ICApOlxuICAgICAgICAgICAgcmV0dXJuIEdwdU1ldGFkYXRhKFwidW5hdmFpbGFibGVcIiwgXCJpbnZhbGlkX3F1'
    'ZXJ5X2Zvcm1hdFwiKVxuICAgICAgICByZXR1cm4gR3B1TWV0YWRhdGEoXCJ2aXNpYmxlX21ldGFkYXRhX29ubHlc'
    'IiwgXCJub3Rfa2VybmVsX2V4ZWN1dGlvblwiLCBmaWVsZHNbMF0sXG4gICAgICAgICAgICAgICAgICAgICAgICAg'
    'ICBpbnQoZmllbGRzWzFdKSwgZmllbGRzWzJdKVxuICAgIGV4Y2VwdCBzdWJwcm9jZXNzLlRpbWVvdXRFeHBpcmVk'
    'OlxuICAgICAgICByZXR1cm4gR3B1TWV0YWRhdGEoXCJ1bmF2YWlsYWJsZVwiLCBcInF1ZXJ5X3RpbWVvdXRcIilc'
    'biAgICBleGNlcHQgKE9TRXJyb3IsIFVuaWNvZGVFcnJvcik6XG4gICAgICAgIHJldHVybiBHcHVNZXRhZGF0YShc'
    'InVuYXZhaWxhYmxlXCIsIFwicXVlcnlfdW5hdmFpbGFibGVcIilcblxuXG5kZWYgaW52ZW50b3J5KCosIGNoZWNr'
    'X2dwdTogYm9vbCA9IEZhbHNlLCB3b3JrZGlyOiBQYXRoIHwgTm9uZSA9IE5vbmUpIC0+IGRpY3Q6XG4gICAgXCJc'
    'IlwiT2JzZXJ2ZSBpbmRlcGVuZGVudCBjb21wb25lbnRzOyByZXNvdXJjZSBmYWlsdXJlIGRvZXMgbm90IHNraXAg'
    'dGhlIEdQVSBxdWVyeS5cIlwiXCJcbiAgICBpZiB0eXBlKGNoZWNrX2dwdSkgaXMgbm90IGJvb2w6XG4gICAgICAg'
    'IHJhaXNlIFZhbHVlRXJyb3IoXCJleHBsaWNpdCBib29sZWFuIEdQVSBxdWVyeSBjaG9pY2UgcmVxdWlyZWRcIilc'
    'biAgICBwYWNrYWdlcyA9IHt9XG4gICAgZm9yIG5hbWUgaW4gUEFDS0FHRVM6XG4gICAgICAgIHRyeTpcbiAgICAg'
    'ICAgICAgIHBhY2thZ2VzW25hbWVdID0gaW1wb3J0bGliLm1ldGFkYXRhLnZlcnNpb24obmFtZSlcbiAgICAgICAg'
    'ZXhjZXB0IGltcG9ydGxpYi5tZXRhZGF0YS5QYWNrYWdlTm90Rm91bmRFcnJvcjpcbiAgICAgICAgICAgIHBhY2th'
    'Z2VzW25hbWVdID0gXCJ1bmF2YWlsYWJsZVwiXG4gICAgb2JzZXJ2ZWQgPSB7XCJweXRob25cIjogcGxhdGZvcm0u'
    'cHl0aG9uX3ZlcnNpb24oKSwgKipwYWNrYWdlc31cbiAgICBleHBlY3RlZCA9IHsqKkVYUEVDVEVELCBcInRvcmNo'
    'XCI6IFwiMi4xMS4wK2N1MTI4XCIgaWYgY2hlY2tfZ3B1IGVsc2UgXCIyLjExLjArY3B1XCJ9XG4gICAgbWF0Y2hl'
    'cyA9IGFsbChvYnNlcnZlZC5nZXQoaykgPT0gdiBmb3IgaywgdiBpbiBleHBlY3RlZC5pdGVtcygpKVxuICAgIGhv'
    'c3QgPSByZXNvdXJjZXMoKVxuICAgIGNhcmQgPSBncHUod29ya2Rpcj13b3JrZGlyKSBpZiBjaGVja19ncHUgZWxz'
    'ZSBHcHVNZXRhZGF0YShcIm5vdF9yZXF1ZXN0ZWRcIiwgXCJjcHVfZGVmYXVsdFwiKVxuICAgIGlmIG5vdCBtYXRj'
    'aGVzOlxuICAgICAgICBzdGF0dXMgPSBcInVuYXZhaWxhYmxlX2Vudmlyb25tZW50XCJcbiAgICBlbGlmIGNoZWNr'
    'X2dwdSBhbmQgY2FyZC5zdGF0dXMgIT0gXCJ2aXNpYmxlX21ldGFkYXRhX29ubHlcIjpcbiAgICAgICAgc3RhdHVz'
    'ID0gXCJ1bmF2YWlsYWJsZV9ncHVcIlxuICAgIGVsaWYgaG9zdC5zdGF0dXMgIT0gXCJvYnNlcnZlZF9tZXRhZGF0'
    'YV9vbmx5XCI6XG4gICAgICAgIHN0YXR1cyA9IFwicGFydGlhbF9yZXNvdXJjZXNfdW5hdmFpbGFibGVcIlxuICAg'
    'IGVsc2U6XG4gICAgICAgIHN0YXR1cyA9IFwibWV0YWRhdGFfcmVhZHlfb25seVwiXG4gICAgcmV0dXJuIHtcbiAg'
    'ICAgICAgXCJzY2hlbWFcIjogXCJzcGF0aWFsLWNvbGFiLXJlYWRpbmVzcy12M1wiLCBcInN0YXR1c1wiOiBzdGF0'
    'dXMsXG4gICAgICAgIFwicHl0aG9uXCI6IG9ic2VydmVkW1wicHl0aG9uXCJdLCBcInBhY2thZ2VzXCI6IHBhY2th'
    'Z2VzLCBcInBhY2thZ2VfbWV0YWRhdGFfbWF0Y2hlc1wiOiBtYXRjaGVzLFxuICAgICAgICBcInJlc291cmNlc1wi'
    'OiBhc2RpY3QoaG9zdCksIFwiZ3B1XCI6IGFzZGljdChjYXJkKSwgXCJyZXNvdXJjZV9lbmZvcmNlbWVudFwiOiBc'
    'InVudmVyaWZpZWRcIixcbiAgICAgICAgXCJwcm9jZXNzX2xpbWl0c1wiOiB7XG4gICAgICAgICAgICBuYW1lOiBs'
    'aXN0KHJlc291cmNlLmdldHJsaW1pdChrZXkpKVxuICAgICAgICAgICAgZm9yIG5hbWUsIGtleSBpbiAoKFwiYWRk'
    'cmVzc19zcGFjZVwiLCByZXNvdXJjZS5STElNSVRfQVMpLFxuICAgICAgICAgICAgICAgICAgICAgICAgICAgICAg'
    'KFwiY3B1X3NlY29uZHNcIiwgcmVzb3VyY2UuUkxJTUlUX0NQVSksXG4gICAgICAgICAgICAgICAgICAgICAgICAg'
    'ICAgICAoXCJmaWxlX2J5dGVzXCIsIHJlc291cmNlLlJMSU1JVF9GU0laRSkpXG4gICAgICAgIH0sXG4gICAgICAg'
    'IFwibW9kZWxzX2NyZWF0ZWRcIjogMCwgXCJvcHRpbWl6ZXJfdXBkYXRlc1wiOiAwLCBcInRlbnNvcl9vcGVyYXRp'
    'b25zXCI6IDAsXG4gICAgICAgIFwidG9yY2hfaW1wb3J0ZWRfYnlfc2VydmljZVwiOiBGYWxzZSwgXCJwcm9maWxl'
    'X2F0dGVtcHRfc3RhcnRlZFwiOiBGYWxzZSxcbiAgICB9XG5cblxuZGVmIHJlY29yZChvdXQ6IFBhdGgsICosIGNo'
    'ZWNrX2dwdTogYm9vbCA9IEZhbHNlKSAtPiBkaWN0OlxuICAgIFwiXCJcIkV4Y2x1c2l2ZSBtZXRhZGF0YSBldmlk'
    'ZW5jZSwgd2l0aG91dCBhbGxvY2F0aW5nIGEgcnVudGltZSBvciBhdXRob3JpemluZyBmaXR0aW5nLlwiXCJcIlxu'
    'ICAgIGlmIHR5cGUoY2hlY2tfZ3B1KSBpcyBub3QgYm9vbDpcbiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihcImV4'
    'cGxpY2l0IGJvb2xlYW4gR1BVIHF1ZXJ5IGNob2ljZSByZXF1aXJlZFwiKVxuICAgIHN0YXJ0ZWQgPSB0aW1lLm1v'
    'bm90b25pYygpXG4gICAgYS5kaXJlY3Rvcnkob3V0KVxuICAgIGEucHV0KG91dCAvIFwiYXR0ZW1wdC5qc29uXCIs'
    'IGEuZW5jb2RlKHtcbiAgICAgICAgXCJzY2hlbWFcIjogXCJzcGF0aWFsLWNvbGFiLXJlYWRpbmVzcy1hdHRlbXB0'
    'LXYzXCIsIFwiY2hlY2tfZ3B1XCI6IGNoZWNrX2dwdSxcbiAgICAgICAgXCJtYXhfc2Vjb25kc1wiOiBNQVhfU0VD'
    'T05EUywgXCJyZXRyeVwiOiBGYWxzZSxcbiAgICB9KSlcbiAgICByZXN1bHQgPSBpbnZlbnRvcnkoY2hlY2tfZ3B1'
    'PWNoZWNrX2dwdSwgd29ya2Rpcj1vdXQpXG4gICAgcmVzdWx0W1wic2Vjb25kc1wiXSA9IHRpbWUubW9ub3Rvbmlj'
    'KCkgLSBzdGFydGVkXG4gICAgaWYgcmVzdWx0W1wic2Vjb25kc1wiXSA+IE1BWF9TRUNPTkRTOlxuICAgICAgICBy'
    'ZXN1bHRbXCJzdGF0dXNcIl0gPSBcInVuYXZhaWxhYmxlX2RlYWRsaW5lXCJcbiAgICByZXN1bHRbXCJyZWNlaXB0'
    'X3NoYTI1NlwiXSA9IGEucHV0KG91dCAvIFwicmVzdWx0Lmpzb25cIiwgYS5lbmNvZGUocmVzdWx0KSwgTUFYX09V'
    'VFBVVF9CWVRFUylcbiAgICByZXR1cm4gcmVzdWx0XG4iCiAgICB9CiAgfSwKICAic2NoZW1hIjogImNvbGFiLXJl'
    'YWRpbmVzcy1zb3VyY2UtdjMiCn0K'
)


def install_sources(raw, root):
    allowed = {"core/__init__.py", "core/colab_artifacts.py",
               "core/spatial_colab_readiness.py", "LICENSE"}
    if len(raw) > 262144 or hashlib.sha256(raw).hexdigest() != SOURCE_SHA256:
        raise ValueError("source package hash/size mismatch")
    value = json.loads(raw)
    if (set(value) != {"schema", "base_commit", "files"}
            or value["schema"] != "colab-readiness-source-v3"
            or value["base_commit"] != BASE_COMMIT or set(value["files"]) != allowed):
        raise ValueError("source package schema mismatch")
    prepared = {}
    for name, entry in value["files"].items():
        if set(entry) != {"utf8", "sha256"}:
            raise ValueError("source entry schema mismatch")
        content = entry["utf8"].encode("utf-8")
        if len(content) > 65536 or hashlib.sha256(content).hexdigest() != entry["sha256"]:
            raise ValueError("source file hash/size mismatch")
        prepared[name] = content
    if any(p.is_symlink() for p in (root, *root.parents)):
        raise ValueError("source directory symlink")
    root.mkdir(mode=0o700)
    (root / "core").mkdir(mode=0o700)
    for name, content in prepared.items():
        fd = os.open(root / name, os.O_WRONLY | os.O_CREAT | os.O_EXCL | os.O_NOFOLLOW, 0o600)
        with os.fdopen(fd, "wb") as stream:
            stream.write(content)
    return root


if any(name == "core" or name.startswith("core.") for name in sys.modules):
    raise RuntimeError("use a fresh kernel; an existing core module is already loaded")
source_root = install_sources(base64.b64decode(SOURCE_B64, validate=True),
                              Path("/content/spatial-colab-readiness-v3-source-20261106"))
sys.path.insert(0, str(source_root))
print("Verified readiness source package:", SOURCE_SHA256)


In [ ]:
from core.spatial_colab_readiness import record

CHECK_GPU = False  # CPU metadata is the default. No runtime is allocated by this code.
result = record(Path("/content/spatial-colab-readiness-v3-result-20261106"),
                check_gpu=CHECK_GPU)
print(json.dumps(result, indent=2, sort_keys=True))


## Çıktı

`partial_resources_unavailable`: sürümler eşleşti, kaynak metadatası eksik.
`metadata_ready_only`: istenen metadata okunabildi; süre/RAM/swap denetimi henüz doğrulanmış sayılmaz.
`unavailable_environment`: seçilen paket sürümleri beklenen CPU/GPU ortamıyla eşleşmedi.

Mevcut WSL eğitim denetleyicisi ve önceki denemeler değiştirilmez.
Bu kontrol yeni eğitim veya eski denemeyi tekrarlama izni değildir.
Çıktıyı kaydedip CPU oturumunu kapatın.
